In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [ ]:
# ==========================================
# AIMO3 GOLD FINAL NOTEBOOK (FULL SYSTEM)
# ==========================================

# =========================
# 1. Imports & Config
# =========================
import os, re, time, math, random
import numpy as np
import pandas as pd
import sympy as sp
import torch
from collections import defaultdict, Counter
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATHS = [
    "Qwen/Qwen2.5-Math-7B-Instruct",
    "deepseek-ai/DeepSeek-Math-7B-Instruct"
]

MAX_TIME = 60 * 60 * 9

# =========================
# 2. Load Data
# =========================
def load_test():
    paths = [
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",
        "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv"
    ]
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p)
    raise Exception("test.csv not found")

test_df = load_test()
problem_col = [c for c in test_df.columns if c != "id"][0]

# =========================
# 3. Cache
# =========================
CACHE = {}

def get_cache(problem):
    return CACHE.get(problem, None)

def set_cache(problem, result):
    CACHE[problem] = result

# =========================
# 4. Difficulty
# =========================
def estimate_difficulty(p):
    if len(p) > 300:
        return "hard"
    if any(k in p.lower() for k in ["prove", "geometry"]):
        return "hard"
    if any(k in p.lower() for k in ["find", "compute"]):
        return "medium"
    return "easy"

BUDGET = {
    "easy": (1, [0.3, 0.7], 8),
    "medium": (2, [0.3, 0.7, 1.0], 16),
    "hard": (2, [0.3, 0.7, 1.0], 32),
}

# =========================
# 5. Problem Classification
# =========================
def classify(p):
    p = p.lower()
    if "mod" in p or "remainder" in p:
        return "number_theory"
    if "triangle" in p or "circle" in p:
        return "geometry"
    if "ways" in p or "count" in p:
        return "combinatorics"
    return "algebra"

# =========================
# 6. Prompt Mixer
# =========================
def build_prompt(problem, category, style):
    if style == "formal":
        return f"Solve rigorously ({category}). Final \\boxed{{integer}}.\n{problem}"
    if style == "code":
        return f"Use Python for calculations.\n{problem}"
    if style == "search":
        return f"Try brute force if needed.\n{problem}"
    return f"Solve step by step.\n{problem}"

STYLES = ["formal", "code", "search"]

# =========================
# 7. Model Wrapper
# =========================
class Model:
    def __init__(self, path):
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForCausalLM.from_pretrained(
            path, torch_dtype=torch.float16, device_map="auto"
        )

    def generate(self, prompt, temp):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(DEVICE)
        out = self.model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=temp,
            do_sample=True
        )
        return self.tokenizer.decode(out[0], skip_special_tokens=True)

MODELS = [Model(p) for p in MODEL_PATHS]

# =========================
# 8. Extract Answer
# =========================
def extract(text):
    boxed = re.findall(r'\\boxed\{(\d+)\}', text)
    if boxed:
        return [int(x) % 100000 for x in boxed]
    nums = re.findall(r'\b\d{1,5}\b', text[-200:])
    return [int(x) for x in nums]

# =========================
# 9. Safe Exec
# =========================
def exec_code(code):
    try:
        loc = {}
        exec(code, {"math":math, "sp":sp}, loc)
        return str(loc)
    except:
        return ""

# =========================
# 10. Deep TIR
# =========================
def deep_tir(model, prompt, temp, depth=3):
    text = model.generate(prompt, temp)
    
    for _ in range(depth):
        codes = re.findall(r'```python(.*?)```', text, re.DOTALL)
        for c in codes[:2]:
            out = exec_code(c)
            text += "\nOutput:\n" + out
        
        text += model.generate("Continue reasoning", 0.3)
    
    return text

# =========================
# 11. Number Theory Solver
# =========================
def crt_solver(problem):
    results = []
    
    # brute mod search
    for x in range(0, 1000):
        try:
            if eval(problem.replace("=", "==")):
                results.append(x)
        except:
            continue
    
    return results

# =========================
# 12. Voting
# =========================
def vote(candidates):
    cnt = Counter(candidates)
    if not cnt:
        return 0, 0
    
    best = cnt.most_common(1)[0]
    conf = best[1] / sum(cnt.values())
    return best[0], conf

# =========================
# 13. Solver
# =========================
def solve(problem):
    
    # cache
    cached = get_cache(problem)
    if cached:
        return cached
    
    diff = estimate_difficulty(problem)
    mcount, temps, samples = BUDGET[diff]
    
    category = classify(problem)
    candidates = []
    
    for m in MODELS[:mcount]:
        for style in STYLES:
            prompt = build_prompt(problem, category, style)
            
            for t in temps:
                for _ in range(samples // len(STYLES)):
                    
                    if diff == "hard":
                        text = deep_tir(m, prompt, t, depth=3)
                    else:
                        text = m.generate(prompt, t)
                    
                    candidates += extract(text)
    
    # number theory fallback
    if category == "number_theory":
        candidates += crt_solver(problem)
    
    ans, conf = vote(candidates)
    
    result = {"answer": ans, "confidence": conf}
    set_cache(problem, result)
    
    return result

# =========================
# 14. Run
# =========================
results = []
start = time.time()

for i, row in test_df.iterrows():
    pid = row["id"]
    prob = row[problem_col]
    
    res = solve(prob)
    
    results.append({
        "id": pid,
        "answer": int(res["answer"])
    })
    
    if (time.time() - start) > MAX_TIME:
        break

sub = pd.DataFrame(results)
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("DONE")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.8/478.8 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
google-adk 1.25.1 requires opentelemetry-api<1.40.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.25.1 requires opentelemetry-sdk<1.40.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1

✅ Dependencies installed
✅ Imports complete
✅ Config loaded | primary=transformers


FileNotFoundError: test.csv not found — check competition data sources